In [627]:
import math as m
import numpy as np
import random 
import stable_baselines3
#import gym 
import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F
import random 

from abc import ABC, abstractmethod



In [628]:
import torch
import torch.nn as nn

print(torch.__version__)

net = nn.Linear(4, 4)
#optimizer = torch.optim.SGD(net.parameters(), lr=3e-4)

print("works")

2.13.0+cu130
works


## Building the Env

In [629]:
# redefine box and env  as in box is singular isolated , part of a bigger formula , pattern , structure made by the bigger class .
#test the stuff here 

class CellBox :
    def __init__(
            self,
            name,
            xy:tuple[int,int],
            color:str, 
            symbol:str=None, 
            dirlist : list = None, 
            reward_fn = None
        ):

        # this part describes the box itself
        self.name=name
        self.coordinate=xy
        self.color=color
        self.reward=0
        
        self.reward_fn = reward_fn if reward_fn != None else self.default_rewardfn
        
        self.symbol= symbol
        #this one is about the surrounding of the box
        self.neighbours={
            "up":dirlist[0],
            "down":dirlist[1],
            "right":dirlist[2],
            "left":dirlist[3],
            "stand":self
        }

    def default_rewardfn(self):
        if self.symbol == "X":
            self.reward = -100
        elif self.symbol == "O":
            self.reward = 100
        else:
            self.reward = -10

        return self.reward

    def reward_(self):
        return  self.reward_fn()

    def next_state(self,action):
        if self.neighbours[action] is None:
            return self
        return self.neighbours[action] 
    
    # 🔥 KEY FIX: equality based on identity of state meaning
    def __eq__(self, other):
        return isinstance(other, CellBox) and self.name == other.name

    def __hash__(self):
        return hash(self.name)

In [630]:
class Board(ABC): # i could ask for completion and correction on this board , but i think it is not important
                   # basically not a priority
    def __init__(self,
                 cells: list[CellBox],
                 geometry="box",
                 data=None):

        self.cells = cells
        self.geometry = geometry
        self.data = data

        self.form()

    def form(self):

        if self.geometry == "box":
            self.make_box(self.data)

        elif self.geometry == "pyramid":
            self.make_pyramid(self.data)

        elif self.geometry == "stack":
            self.make_stack(self.data)

        elif self.geometry == "h_line":
            self.make_line({"direction": "h"})

        elif self.geometry == "v_line":
            self.make_line({"direction": "v"})

        elif self.geometry == "custom":
            self.make_custom(self.data)

    # -------------------------------------------------------

    def __iter__(self):
        return iter(self.cells)

    def __len__(self):
        return len(self.cells)

    def snapshot(self):
        return self.cells.copy()

    # -------------------------------------------------------

    def make_stack(self, data):
        pass

    def make_box(self, data):

        shape = data["shape"]
        N, M = shape

        cells = self.cells

        for m in range(M):
            for n in range(N):

                p = n + m * N
                s = cells[p]

                s.neighbours = {
                    "up": None,
                    "down": None,
                    "right": None,
                    "left": None,
                    "stand": s
                }

                if n < N - 1:
                    s.neighbours["right"] = cells[p + 1]

                if n > 0:
                    s.neighbours["left"] = cells[p - 1]

                if m < M - 1:
                    s.neighbours["up"] = cells[p + N]

                if m > 0:
                    s.neighbours["down"] = cells[p - N]

    def make_pyramid(self, data):
        pass

    def make_custom(self, data): 
        
        opposite = {
            "right": "left",
            "left":  "right",
            "up":    "down",
            "down":  "up",
            }
        # Reset every cell first
        for cell in self.cells:

            cell.neighbours = {
                "up": None,
                "down": None,
                "left": None,
                "right": None,
                "stand": cell,
            }

        # Apply all user-defined connections
        for cell_a, direction, cell_b in data["connections"]:

            if direction not in opposite:
                raise ValueError(
                    f"Unknown direction '{direction}'. "
                    f"Allowed: {list(opposite.keys())}"
                )

            # Forward connection
            cell_a.neighbours[direction] = cell_b

            # Reverse connection
            cell_b.neighbours[opposite[direction]] = cell_a

    def make_line(self, data):

        direction = data["direction"]

        if direction == "h":

            for i in range(len(self.cells) - 1):

                self.cells[i].neighbours["right"] = self.cells[i + 1]
                self.cells[i + 1].neighbours["left"] = self.cells[i]

        elif direction == "v":

            for i in range(len(self.cells) - 1):

                self.cells[i].neighbours["up"] = self.cells[i + 1]
                self.cells[i + 1].neighbours["down"] = self.cells[i]

In [631]:
# Env (organizing the puzzle of state into env )
class Env:

    @property
    def actions(self) -> list:
        """All actions available in this environment."""
        raise NotImplementedError

    @property
    def states(self) -> list:
        """All states in this environment."""
        raise NotImplementedError

    def reset(self):
        """Start a new episode. Return the initial state."""
        raise NotImplementedError

    def step(self, action):
        """
        Apply action to the current state.
        Return (next_state, reward, done).
        """
        raise NotImplementedError
    

In [632]:
class Chessboard(Env):
    """
    RL environment built on top of a Board.

    Board
        -> geometry

    Environment
        -> transitions
        -> rewards
        -> terminal conditions
        -> current state
    """

    def __init__(self,
                 board: Board,
                 actions: list[str],
                 start: CellBox):

        self.board = board

        self._actions = actions

        self.start = start
        self.current = start
        # insert self.path = [] saving all the exps that the agent went through

    # -------------------------------------------------------

    @property
    def states(self):
        return self.board.cells

    @property
    def actions(self):
        return self._actions

    # -------------------------------------------------------

    def reset(self):

        self.current = self.start
        return self.current

    # -------------------------------------------------------

    def step(self, action):

        next_state = self.current.next_state(action)

        next_state.reward_()

        reward = float( next_state.reward )

        done = next_state.symbol in ("X", "O")

        self.current = next_state

        return {
            "state": self.current,
            "action": action,
            "reward": reward,
            "next_state": next_state,
            "done": done,
        }

    # -------------------------------------------------------

    def snapshot(self):

        return {
            "current": self.current,
            "start": self.start,
            "board": self.board.snapshot(),
        }

### Setting an example board

In [633]:
#set up
s1 = CellBox("s1",(0,0),"white","X",[None,None,None,None])
s2 = CellBox("s2",(2,0),"white","O",[None,None,None,None])
s3 = CellBox("s3",(4,0),"white","X",[None,None,None,None])
s4 = CellBox("s4",(0,1),"white",None,[None,None,None,None])
s5 = CellBox("s5",(1,1),"grey",None,[None,None,None,None])
s6 = CellBox("s6",(2,1),"white",None,[None,None,None,None])
s7 = CellBox("s7",(3,1),"grey",None,[None,None,None,None])
s8 = CellBox("s8",(4,1),"white",None,[None,None,None,None])

#s0 = CellBox("empty box")

#define
s1.neighbours = {"up": s4,   "down": None, "right": None, "left": None, "stand": s1}
s2.neighbours = {"up": s6,   "down": None, "right": None, "left": None, "stand": s2}
s3.neighbours = {"up": s8,   "down": None, "right": None, "left": None, "stand": s3}
s4.neighbours = {"up": None, "down": s1,   "right": s5,   "left": None, "stand": s4}
s5.neighbours = {"up": None, "down": None, "right": s6,   "left": s4,   "stand": s5}
s6.neighbours = {"up": None, "down": s2,   "right": s7,   "left": s5,   "stand": s6}
s7.neighbours = {"up": None, "down": None, "right": s8,   "left": s6,   "stand": s7}
s8.neighbours = {"up": None, "down": s3,   "right": None, "left": s7,   "stand": s8}
#fwefsdf
States1=[s1,s2,s3,s4,s5,s6,s7,s8]

Actions=["up", "right","down","left"]




In [634]:
s1 = CellBox("s1",(2,0),"white",None,[None,None,None,None])
s2 = CellBox("s2",(2,1),"white",None,[None,None,None,None])
s3 = CellBox("s3",(2,2),"white",None,[None,None,None,None])
s4 = CellBox("s4",(1,0),"white",None,[None,None,None,None])
s5 = CellBox("s5",(1,1),"white","X",[None,None,None,None])
s6 = CellBox("s6",(1,2),"white",None,[None,None,None,None])
s7 = CellBox("s7",(0,0),"white",None,[None,None,None,None])
s8 = CellBox("s8",(0,1),"white",None,[None,None,None,None])
s9 = CellBox("s9",(0,2),"white","O",[None,None,None,None])

States2 = [s1,s2,s3,s4,s5,s6,s7,s8,s9]

claude_board = Board(States2,geometry="box",data={"shape":(3,3)})

env = Chessboard(
    board=claude_board,
    actions=Actions,
    start=s1
)

state = env.reset()

experience = env.step("right")

In [635]:
experience["next_state"].name

isinstance(env, Env)

True

## Define the State vector

In [636]:
#reorganize the whole code here

def manhattan_goal(state:CellBox):
        x,y = state.coordinate
        return abs(x-0) + abs(y-2)

def manhattan_trap(state:CellBox):
        x,y = state.coordinate
        return abs(x-1) + abs(y-1)


#the state _ vector
def state_to_vector(state):

    color = 0 if state.color == "white" else 1

    if state.symbol == "X":
        symbol = -1
    elif state.symbol == "O":
        symbol = 1
    else:
        symbol = 0

    return np.array([
        state.coordinate[0],
        state.coordinate[1],
        color,
        symbol,
        manhattan_goal(state=state),
        manhattan_trap(state=state)
    ], dtype=np.float32)

## Policy

In [637]:
from abc import ABC, abstractmethod

class Policy(ABC):
    @abstractmethod
    def select_action(self, state): pass
    @abstractmethod
    def action_distribution(self, state): pass
    @abstractmethod
    def snapshot(self): pass

### Some examples

In [638]:
class DeterministicPolicy(Policy):
    def __init__(self):
        self.mapping = {}
    def select_action(self, state):        return self.mapping[state]
    def action_distribution(self, state):  return {self.mapping[state]: 1.0}
    def snapshot(self):                    return self.mapping.copy()

class StochasticPolicy(Policy):
    def __init__(self):
        self.pi = {}
    def select_action(self, state):
        actions = list(self.pi[state].keys())
        probs   = list(self.pi[state].values())
        return random.choices(actions, probs)[0]
    def action_distribution(self, state):  return self.pi[state]
    def snapshot(self):                    return self.pi.copy()

class Policy_theta_softmax(Policy):
    def __init__(self, phi, theta): # phi:FeatureFunction
        self.pi      = {}
        self.weights = theta
        self.phi     = phi

    def _zeller(self, state, action, Actions): #
        n  = Actions.index(action)
        a1 = Actions[(n + 1) % 4]
        a2 = Actions[(n + 3) % 4]
        phi_sa = 0.8 * self.phi.compute(state.next_state(action)) + \
                 0.1 * (self.phi.compute(state.next_state(a1)) +
                        self.phi.compute(state.next_state(a2)))
        h = (self.weights @ phi_sa).item()
        return np.exp(h)

    def calculate_all_probabilities(self, state, Actions):
        # compute each zeller once, reuse for denominator
        zellers = {a: self._zeller(state, a, Actions) for a in Actions}
        N = sum(zellers.values())
        self.pi[state] = {a: float(z / N) for a, z in zellers.items()}

    def select_action(self, state):
        actions = list(self.pi[state].keys())
        probs   = list(self.pi[state].values())
        return random.choices(actions, probs)[0]

    def action_distribution(self, state):  return self.pi.get(state, {})
    def snapshot(self):                    return self.pi.copy()

class NeuralPolicy(Policy):
    def __init__(self, network):
        self.network = network
    def select_action(self, state):
        probs = self.network(state)
        return 0  # placeholder
    def action_distribution(self, state):  return self.network(state)
    def snapshot(self):                    return self.network.state_dict()

## Learning Strategy (for improvement and update)

In [639]:
from abc import ABC, abstractmethod

class LearningStrategy(ABC):
    @abstractmethod
    def update(self, policy, experience): pass
    @abstractmethod
    def snapshot(self): pass

class QLearning(LearningStrategy):
    def __init__(self, actions, alpha=0.1, gamma=0.9):
        self.alpha   = alpha
        self.gamma   = gamma
        self.actions = actions
        self.Q       = {}

    def _ensure(self, s):
        if s not in self.Q:
            self.Q[s] = {a: 0.0 for a in self.actions}

    def update(self, policy, exp):
        s, a, r, next_s, done = exp["state"], exp["action"], exp["reward"], exp["next_state"], exp["done"]
        self._ensure(s)
        self._ensure(next_s)
        target = r if done else r + self.gamma * max(self.Q[next_s].values())
        self.Q[s][a] += self.alpha * (target - self.Q[s][a])
        policy.mapping[s] = max(self.Q[s], key=self.Q[s].get)

    def snapshot(self):
        return self.Q.copy()

class Reinforce(LearningStrategy):
    def __init__(self, lr=0.1):
        self.lr = lr
    def update(self, policy, state_vec, action_index, reward):
        probs = policy.forward(state_vec)
        for i in range(policy.W.shape[1]):
            policy.W[0, i] += self.lr * reward * ((1 if i == action_index else 0) - probs[i]) * state_vec[0]
    def snapshot(self): return {}



## Generalized Advantage Estimation

In [640]:
def compute_gae(deltas, gamma=0.99, lam=0.95):
    """Compute GAE advantages from a list of TD residuals (deltas)"""
    advantages = torch.zeros(len(deltas))
    gae = torch.tensor(0.0)
    
    # Backward pass - this is the key
    for t in reversed(range(len(deltas))):
        gae = deltas[t] + gamma * lam * gae
        advantages[t] = gae
    
    return advantages



### example


In [641]:
# Simulate adding steps one by one (as in your question)
deltas = []

print("Incremental GAE Update:\n")

for step in range(1, 7):
    # Simulate receiving a new step
    new_delta = round(np.random.uniform(-1.0, 2.0), 3)   # random for demo
    deltas.append(new_delta)
    
    print(f"After step {step} (new δ = {new_delta}):")
    
    # Recompute GAE using ALL data available so far
    advantages = compute_gae(deltas, gamma=0.99, lam=0.95)
    
    for i, adv in enumerate(advantages):
        print(f"   A[{i+1:2d}] = {adv:.3f}")
    print("-" * 50)

Incremental GAE Update:

After step 1 (new δ = 1.566):
   A[ 1] = 1.566
--------------------------------------------------
After step 2 (new δ = 0.636):
   A[ 1] = 2.164
   A[ 2] = 0.636
--------------------------------------------------
After step 3 (new δ = 1.789):
   A[ 1] = 3.747
   A[ 2] = 2.319
   A[ 3] = 1.789
--------------------------------------------------
After step 4 (new δ = 1.168):
   A[ 1] = 4.718
   A[ 2] = 3.352
   A[ 3] = 2.888
   A[ 4] = 1.168
--------------------------------------------------
After step 5 (new δ = -0.771):
   A[ 1] = 4.115
   A[ 2] = 2.710
   A[ 3] = 2.206
   A[ 4] = 0.443
   A[ 5] = -0.771
--------------------------------------------------
After step 6 (new δ = -0.31):
   A[ 1] = 3.887
   A[ 2] = 2.468
   A[ 3] = 1.948
   A[ 4] = 0.169
   A[ 5] = -1.063
   A[ 6] = -0.310
--------------------------------------------------


## Blocks to build Neural Network

In [642]:
# Uncomment if you want to use this simple version Brick 1 and 2 

# ── BRICK 1: reusable block (your Block, kept exactly) ──────────────────
class Block(nn.Module):
    """A reusable residual-style chunk: linear → norm → activation"""
    def __init__(self, dim, activation=nn.ReLU):
        super().__init__()
        self.fc   = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)
        self.act  = activation()

    def forward(self, x):
        return self.act(self.norm(self.fc(x)))

# ── BRICK 2: shared body ─────────────────────────────────────────────────
class SharedBody(nn.Module):
    """
    Encodes the state φ(s) into a shared representation.
    Both actor and critic read from this.
    """
    def __init__(
        self,
        n_features: int,
        hidden_dim: int = 64,
        n_layers: int = 2,
        activation=nn.ReLU
    ):
        super().__init__()

        self.input_layer = nn.Linear(n_features, hidden_dim)

        layers = [
            self.input_layer,
            activation()
        ]

        for i in range(1, n_layers + 1):
            setattr(self, f"block{i}", Block(hidden_dim, activation))

        for i in range(1, n_layers + 1):
            layers.append(getattr(self, f"block{i}"))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


# class SharedBody(nn.Module): # efficient version
#     def __init__(
#         self,
#         n_layers: int,
#         n_features: int,
#         hidden_dim: int = 64,
#         activation=nn.ReLU
#     ):
#         super().__init__()

#         self.network = nn.Sequential(
#             nn.Linear(n_features, hidden_dim),
#             activation(),
#             *[Block(hidden_dim, activation) for _ in range(n_layers)]
#         )

        
#vars() was useful here 

In [643]:
# # Uncomment if you want to use this simple version Brick 1 and 2 

#── BRICK 1: reusable block (your Block, kept exactly) ──────────────────
# class Block(nn.Module):
#     """A reusable residual-style chunk: linear → norm → relu"""
#     def __init__(self, dim):
#         super().__init__()
#         self.fc   = nn.Linear(dim, dim)
#         self.norm = nn.LayerNorm(dim)

#     def forward(self, x):
#         return F.relu(self.norm(self.fc(x)))  # ← fixed: F.relu() not nn.ReLU()


#── BRICK 2: shared body ─────────────────────────────────────────────────
# class SharedBody(nn.Module):
#     """
#     Encodes the state φ(s) into a shared representation.
#     Both actor and critic read from this — they see the same features.
#     Input:  φ(s) of shape (n_features,)
#     Output: hidden representation of shape (hidden_dim,)
#     """
#     def __init__(self, n_features: int, hidden_dim: int = 64):
#         super().__init__()
#         self.input_layer = nn.Linear(n_features, hidden_dim)
#         self.block1      = Block(hidden_dim)
#         self.block2      = Block(hidden_dim)

#         #self.network = nn.Sequential(....)

#     def forward(self, x):
#         x = F.relu(self.input_layer(x))
#         x = self.block1(x)
#         x = self.block2(x)
#         return x

In [644]:

# ── BRICK 3: actor head ──────────────────────────────────────────────────
class ActorHead(nn.Module):
    """
    Takes shared body output → outputs a probability distribution over actions.
    π_θ(a|s) = softmax(W · h + b)
    """
    def __init__(self, hidden_dim: int, n_actions: int):
        super().__init__()
        self.head = nn.Linear(hidden_dim, n_actions)

    def forward(self, h):
        return F.softmax(self.head(h), dim=-1)   # shape: (n_actions,)


# ── BRICK 4: critic head ─────────────────────────────────────────────────
class CriticHead(nn.Module):
    """
    Takes shared body output → outputs a single scalar V(s).
    No activation — value can be any real number.
    """
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, h):
        return self.head(h).squeeze(-1)           # shape: scalar


# ── BRICK 5: full Actor-Critic network ───────────────────────────────────
class ActorCritic(nn.Module):
    """
    One network, two outputs:
        actor  → π_θ(a|s)   used for: action selection + policy loss
        critic → V_θ(s)     used for: GAE computation  + value loss
    """
    def __init__(self, n_features: int, n_actions: int, hidden_dim: int = 64,n_layers = 2):
        super().__init__()
        self.body   = SharedBody(n_features, hidden_dim,n_layers)
        self.actor  = ActorHead(hidden_dim, n_actions)
        self.critic = CriticHead(hidden_dim)

    def forward(self, x):
        h     = self.body(x)
        probs = self.actor(h)    # π_θ(a|s)
        value = self.critic(h)   # V_θ(s)
        return probs, value

    def get_action(self, state_vec: np.ndarray):
        """
        Given a raw state vector, returns:
          action   — sampled from π_θ(a|s)
          log_prob — log π_θ(a|s), needed for PPO ratio
          value    — V_θ(s), stored in rollout buffer
        """
        x           = torch.tensor(state_vec, dtype=torch.float32)
        probs, value = self.forward(x)
        dist        = torch.distributions.Categorical(probs)
        action_idx  = dist.sample()
        log_prob    = dist.log_prob(action_idx)
        return action_idx.item(), log_prob, value


# ── BRICK 6: your custom loss (kept from your code) ──────────────────────
class MyLoss(nn.Module):
    """MSE loss — used for critic: (V(s) - y_t)²"""
    def forward(self, y_pred, y_true):
        return ((y_pred - y_true) ** 2).mean()
    

In [645]:
class Policy_PPO(Policy):
    def __init__(self,network,actions):
        super().__init__()
        self.pi ={}
        self.net = network
        self.actions = actions
        #self.log = []
        #self.value = {} # to store the values for each state - defined externally
        # not the primary fn of pi but we can save some computation energy
         
    def select_action(self, state):
        x = state_to_vector(state)
        a_idx , log_prob , value = self.net.get_action(x) # see what we can do with the value
        #self.log.append(log_prob)
        #self.value[state] = value
        return self.actions[a_idx] , log_prob, value

    def action_distribution(self, state):
        x = torch.tensor(state_to_vector(state),dtype= torch.float32)
        probs, value = self.net.forward(x)
        
        return self.net(x)
    def snapshot(self): return self.net.state_dict()
        

### Examples for sanity check

In [646]:
# ── quick sanity check ───────────────────────────────────────────────────
n_features = 2   # [dist_goal, dist_trap] — your phi(s) size
n_actions  = 4   # up, right, down, left

net = ActorCritic(n_features=n_features, n_actions=n_actions, hidden_dim=64, n_layers= 100)

dummy_state = torch.tensor([2.0, 1.0], dtype=torch.float32)
probs, value = net(dummy_state)

print(f"Action probs : {probs.detach().numpy()}")  # 4 numbers summing to 1
print(f"Value        : {value.item():.4f}")        # single scalar
print(f"Total params : {sum(p.numel() for p in net.parameters())}")

# act from a real numpy feature vector
state_vec = np.array([2.0, 1.0])
action_idx, log_prob, val = net.get_action(state_vec) 
print(f"Sampled action index: {action_idx} → {Actions[action_idx]}")

Action probs : [0.39078435 0.16588122 0.1360065  0.3073279 ]
Value        : -0.3283
Total params : 429317
Sampled action index: 1 → right


In [647]:
#vars(net.body)

### example to see if everything works together

In [648]:
example_state = env.states[0]
n_features = len(state_to_vector(example_state))
pi = Policy_PPO(ActorCritic(n_features= n_features , n_actions=n_actions, hidden_dim=64),Actions) 
pi.select_action(s6) #Actions[a_idx] , log_prob, value

('left',
 tensor(-1.1864, grad_fn=<SqueezeBackward1>),
 tensor(-0.3908, grad_fn=<SqueezeBackward1>))

## Defining Loss class for Improvement / Learning / Updating

In [649]:
class Loss(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, *args, **kwargs):
        raise NotImplementedError



## My Loss function for PPO

In [650]:
class Loss_Entropy(Loss):
    """L_H = -Σ p·log(p)  — encourages exploration"""
    def forward(self, probs: torch.Tensor) -> torch.Tensor:
        eps = 1e-8
        return -(probs * torch.log(probs + eps)).sum(dim=-1).mean()

class Loss_Critic(Loss):
    """L_V = (V(s) - y)²  — makes critic predict returns accurately"""
    def forward(self, v_pred: torch.Tensor, v_target: torch.Tensor) -> torch.Tensor:
        #return ((v_pred - v_target) ** 2).mean()
        return F.mse_loss(v_pred, v_target)
    
class Loss_Actor(Loss):
    """L_pi = -ρ·A  — PPO clipped policy gradient"""
    def __init__(self, clip_epsilon=0.2):
        super().__init__()
        self.eps = clip_epsilon

    def forward(self, log_prob_new: torch.Tensor,
                      log_prob_old: torch.Tensor,
                      advantage:   torch.Tensor) -> torch.Tensor:
        # ratio ρ = π_new / π_old  via log difference (numerically stable)
        rho     = torch.exp(log_prob_new - log_prob_old)
        clipped = torch.clamp(rho, 1 - self.eps, 1 + self.eps)
        # take the pessimistic (minimum) of clipped and unclipped
        L_clip  = torch.min(rho * advantage, clipped * advantage)
        return -L_clip.mean()   # negative because we MAXIMIZE, but optimizer MINIMIZES

class Loss_PPO(Loss):
    """Total PPO loss = L_actor + cv·L_critic - ce·L_entropy"""
    def __init__(self, cv=0.5, ce=0.01, clip_epsilon=0.2):
        super().__init__()
        self.cv    = ce
        self.ce    = cv
        self.L_pi  = Loss_Actor(clip_epsilon)
        self.L_v   = Loss_Critic()
        self.L_h   = Loss_Entropy()

    def forward(self,
                log_prob_new: torch.Tensor,
                log_prob_old: torch.Tensor,
                advantage:    torch.Tensor,
                v_pred:       torch.Tensor,
                v_target:     torch.Tensor,
                probs:        torch.Tensor) -> torch.Tensor:

        actor_loss  = self.L_pi(log_prob_new, log_prob_old, advantage)
        critic_loss = self.L_v(v_pred, v_target)
        entropy     = self.L_h(probs)

        return actor_loss + self.cv * critic_loss - self.ce * entropy

In [651]:

class PPO(LearningStrategy):
    def __init__(self,
                optimizer,
                clip_epsilon: float = 0.2,
                cv:           float = 0.5,    # critic loss weight
                ce:           float = 0.01,   # entropy bonus weight
                gamma:        float = 0.99,
                lam:          float = 0.95,
                n_epochs:     int   = 1):
        self.optimizer    = optimizer
        self.clip_epsilon = clip_epsilon
        self.cv           = cv          # ← fixed: was swapped
        self.ce           = ce          # ← fixed: was swapped
        self.gamma        = gamma
        self.lam          = lam
        self.n_epochs     = n_epochs
        self.loss_fn      = Loss_PPO(cv=cv, ce=ce,
                                    clip_epsilon=clip_epsilon)  # ← created once

    def compute_gae(self, deltas: list,
                    gamma: float = None,
                    lam:   float = None) -> torch.Tensor:
        gamma = gamma if gamma is not None else self.gamma
        lam   = lam   if lam   is not None else self.lam
        T     = len(deltas)
        advantages = torch.zeros(T)
        gae        = torch.tensor(0.0)
        for t in reversed(range(T)):
            gae           = deltas[t] + gamma * lam * gae
            advantages[t] = gae.detach()
        return advantages

    def update(self, policy: Policy_PPO, rollout: dict) -> float:
        deltas         = rollout["deltas"]
        states_visited = rollout["states_visited"]
        log_probs_old  = rollout["log_probs_old"]
        actions_taken  = rollout["actions_taken"]
        values_visited = rollout["values_visited"]

        A        = self.compute_gae(deltas)
        v_tensor = torch.stack([v.detach() for v in values_visited])
        returns  = (A + v_tensor).detach()

        #normalize advantages — stabilizes training
        #if A.std() > 1e-8:
        #   A = (A - A.mean()) / (A.std() + 1e-8)

        total_loss = 0.0

        for _ in range(self.n_epochs):
            log_probs_new, values_new, probs_all = [], [], []

            for s, a in zip(states_visited, actions_taken):
                x        = torch.as_tensor(state_to_vector(s), dtype=torch.float32)
                probs, v = policy.net(x)
                dist     = torch.distributions.Categorical(probs)
                log_p    = dist.log_prob(torch.tensor(policy.actions.index(a)))
                log_probs_new.append(log_p)
                values_new.append(v)
                probs_all.append(probs)

            log_probs_new = torch.stack(log_probs_new)
            log_probs_old_t = torch.stack(log_probs_old)
            v_pred        = torch.stack(values_new)
            probs_tensor  = torch.stack(probs_all)

            loss = self.loss_fn(log_probs_new, log_probs_old_t,
                                A, v_pred, returns, probs_tensor)

            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy.net.parameters(), max_norm=0.5)
            self.optimizer.step()
            total_loss += loss.item()

        return total_loss / self.n_epochs

    def snapshot(self) -> dict:
        return {
            "gamma":        self.gamma,
            "lam":          self.lam,
            "clip_epsilon": self.clip_epsilon,
            "cv":           self.cv,
            "ce":           self.ce,
            "lr":           self.optimizer.param_groups[0]["lr"],
    }

In [652]:
def debug_ppo(agent, env, rollout=None):

    print("\n" + "=" * 60)
    print("PPO DEBUG")
    print("=" * 60)

    net = agent.policy.net

    # ====================================================
    # NETWORK OUTPUTS
    # ====================================================

    print("\nNETWORK OUTPUTS:\n")

    for s in env.states:

        x = torch.as_tensor(
            state_to_vector(s),
            dtype=torch.float32
        )

        with torch.no_grad():
            probs, value = net(x)

        print(f"{s.name}")

        print("  value:", value.item())

        print("  probs:", probs.tolist())

        print("  sum(probs):", probs.sum().item())

        print("  has_nan:", torch.isnan(probs).any().item())

        print("  has_inf:", torch.isinf(probs).any().item())

        print()

    # ====================================================
    # PARAMETERS
    # ====================================================

    print("\nPARAMETERS:\n")

    for name, param in net.named_parameters():

        print(name)

        print("  mean:", param.data.mean().item())

        print("  std :", param.data.std().item())

        print("  min :", param.data.min().item())

        print("  max :", param.data.max().item())

        print("  has_nan:", torch.isnan(param).any().item())

        print("  has_inf:", torch.isinf(param).any().item())

        print()

    # ====================================================
    # GRADIENTS
    # ====================================================

    print("\nGRADIENTS:\n")

    for name, param in net.named_parameters():

        if param.grad is None:

            print(name)
            print("  gradient: None")
            print()

            continue

        grad_norm = param.grad.norm().item()

        print(name)

        print("  norm:", grad_norm)

        print("  mean:", param.grad.mean().item())

        print("  std :", param.grad.std().item())

        print("  has_nan:", torch.isnan(param.grad).any().item())

        print("  has_inf:", torch.isinf(param.grad).any().item())

        print()

    # ====================================================
    # ROLLOUT
    # ====================================================

    if rollout is not None:

        print("\nROLLOUT:\n")

        deltas = rollout["deltas"]

        advantages = agent.strategy.compute_gae(deltas)

        values = torch.stack(
            [v.detach() for v in rollout["values_visited"]]
        )

        returns = advantages + values

        print("advantages:")
        print(advantages)

        print()

        print("returns:")
        print(returns)

        print()

        print("mean advantage:", advantages.mean().item())

        print("std advantage:", advantages.std().item())

        print("mean return:", returns.mean().item())

        print("std return:", returns.std().item())

        print()

    print("=" * 60)

## Create the Agent

In [653]:
# ════════════════════════════════════════════════════════════════════
#  BRICK 4 — AGENT
#  The self. Owns the policy (permanent). Borrows the strategy (swappable).
#  Contains no learning logic — purely coordinates the other bricks.
# ════════════════════════════════════════════════════════════════════
    
class Agent:

    def __init__(self, strategy: LearningStrategy, policy: Policy):
        self.policy   = policy    # permanent — never replaced
        self.strategy = strategy  # swappable — plug any algorithm in
        self.memory   = []        # full experience trace

    def act(self, state) -> str:
        """Ask the strategy what to do, passing the policy as context."""
        return self.policy.select_action(state)

    def learn(self, experience:dict): # experience = {"state": "A","action": "right","reward": 1,"next_state": "B","done": False}
        """Tell the strategy what happened; it writes into the policy."""
        state,action,reward ,next_state,done = experience.values()

        self.strategy.update(self.policy, experience)
        self.memory.append((
            state.name, 
            action, 
            reward, 
            next_state.name, 
            done
        ))

    def swap_strategy(self, new_strategy: LearningStrategy):
        """
        Replace the learning algorithm.
        The policy — and everything it has learned — is untouched.
        """
        self.strategy = new_strategy

In [654]:

class Agent_PPO(Agent):

    def __init__(self, strategy: PPO, policy: Policy_PPO):
        super().__init__(strategy, policy)

    def act(self, state):
        return self.policy.select_action(state)

    def learn(self, rollout):
        loss = self.strategy.update(self.policy, rollout)
        self.memory.append({
            "rollout_size": len(rollout["deltas"]), 
            "loss":loss
        })

    def value(self, state):
        x = torch.as_tensor(
            state_to_vector(state),
            dtype=torch.float32
        )

        _, value = self.policy.net(x)
        return value

## Training function 

In [655]:
def training(
    env:            Env,
    agent:          Agent_PPO,
    episodes:       int = 100,
    steps:          int = 100,
    rollout_length: int = 37,
    random_start:   bool = False
):
    # in training(), replace free_states with weighted sampling
    non_terminal = [s for s in env.states if s.symbol not in ("X", "O")]

    # neighbours of traps — states that can accidentally reach a trap
    trap_neighbours = []
    for s in non_terminal:
        for neighbour in s.neighbours.values():
            if neighbour and neighbour.symbol == "X":
                trap_neighbours.append(s)
                break

    # weighted start: oversample trap neighbours so agent learns to avoid them
    start_pool = non_terminal + trap_neighbours * 3  # 3x more likely to start near trap

    # Episode loop
    for episode in range(episodes):

        if random_start:
            env.current = random.choice(start_pool)  # ← only picks from valid starts
        else:
            env.reset()

        s = env.current

        # guard: if somehow start is still terminal, skip episode
        if s.symbol in ("X", "O"):
            continue

        # --- rollout buffer ---
        deltas         = []
        log_probs_old  = []
        states_visited = []
        actions_taken  = []
        values_visited = []

        for step in range(steps):

            # --- agent acts ---
            a, log_prob, v_s = agent.act(s)

            # --- environment responds ---
            exp  = env.step(a)
            n_s  = exp["next_state"]
            r    = exp["reward"]
            done = exp["done"]

            # --- TD residual ---
            with torch.no_grad():
                x       = torch.as_tensor(state_to_vector(n_s), dtype=torch.float32)
                _, v_ns = agent.policy.net(x)

            # correct terminal bootstrap
            delta = (r - v_s.detach()) if done else \
                    (r + agent.strategy.gamma * v_ns - v_s.detach())

            # --- store ---
            deltas.append(delta)
            states_visited.append(s)
            actions_taken.append(a)
            values_visited.append(v_s.detach())
            log_probs_old.append(log_prob.detach())

            s = n_s

            # --- PPO update every rollout_length steps ---
            if (step + 1) % rollout_length == 0:
                agent.learn({
                    "deltas":         deltas,
                    "states_visited": states_visited,
                    "log_probs_old":  log_probs_old,
                    "actions_taken":  actions_taken,
                    "values_visited": values_visited,
                })
                deltas.clear(); log_probs_old.clear()
                states_visited.clear(); actions_taken.clear()
                values_visited.clear()

            # --- episode ended ---
            if done:
                break

        # --- learn whatever is left ---
        if len(deltas) > 0:
            agent.learn({
                "deltas":         deltas,
                "states_visited": states_visited,
                "log_probs_old":  log_probs_old,
                "actions_taken":  actions_taken,
                "values_visited": values_visited,
            })

In [663]:
# ── usage ────────────────────────────────────────────────────────────────
n_features = len(state_to_vector(env.states[0]))   # 6
n_actions  = len(Actions)                           # 4

net       = ActorCritic(n_features=n_features, n_actions=n_actions, hidden_dim=64)
pi        = Policy_PPO(net ,Actions)
optimizer = torch.optim.Adam(net.parameters(), lr=3e-4)
ppo       = PPO(optimizer=optimizer, clip_epsilon=0.2, gamma=0.9, lam=0.95 ,cv = 5.0 , ce= 0.01 ) # lam switch from 0.95 to 0.80
agent     = Agent_PPO(strategy=ppo, policy=pi)

env = Chessboard(board=claude_board, actions=Actions, start=s1)

training(env=env, agent=agent, episodes=5000 , steps = 200 ,rollout_length= 64 , random_start=True)

debug_ppo(agent, env)


PPO DEBUG

NETWORK OUTPUTS:

s1
  value: 51.919795989990234
  probs: [0.40240830183029175, 0.3102864921092987, 0.15833400189876556, 0.12897129356861115]
  sum(probs): 1.0000001192092896
  has_nan: False
  has_inf: False

s2
  value: 55.75753402709961
  probs: [0.05147304758429527, 0.7752532958984375, 0.10402695834636688, 0.06924667209386826]
  sum(probs): 0.9999999403953552
  has_nan: False
  has_inf: False

s3
  value: 78.32675170898438
  probs: [0.5325425267219543, 0.25362688302993774, 0.1159667894244194, 0.09786375612020493]
  sum(probs): 0.9999999403953552
  has_nan: False
  has_inf: False

s4
  value: 55.659671783447266
  probs: [0.6318315863609314, 0.15908363461494446, 0.11272480338811874, 0.0963599681854248]
  sum(probs): 1.0
  has_nan: False
  has_inf: False

s5
  value: 64.41342163085938
  probs: [0.10960882157087326, 0.7163394093513489, 0.10286688804626465, 0.07118485122919083]
  sum(probs): 1.0
  has_nan: False
  has_inf: False

s6
  value: 95.89026641845703
  probs: [0.649

/tmp/ipykernel_2279757/2308656204.py:51: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1861.)
  print("  std :", param.data.std().item())
/tmp/ipykernel_2279757/2308656204.py:87: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1861.)
  print("  std :", param.grad.std().item())


In [657]:
env.reset()

print(f"before the step {env.current.name}")

env.step("up")

print(f"after the step {env.current.name}")

int(200*37/100)

#env.states

before the step s1
after the step s4


74

In [658]:


for episode in range(10):
    print(f"in the episode {episode + 1} : \n")
    env.reset()
    for step in range(10-1):
        s = env.current
        a,log_pi_a_of_s ,v = agent.act(s)
        print(f"the action chosen for {s.name} is {a} \n")
        print(f"the value of that state is {v} \n")

        exp = env.step(a)

        if exp["done"] : 
            if exp["next_state"].symbol == "X" : symbol = "trap" 
            else: symbol = "goal"
            print(f" we reached the state {exp["next_state"].name} which is the {symbol} \n")
            break

in the episode 1 : 

the action chosen for s1 is up 

the value of that state is 59.87788772583008 

the action chosen for s4 is right 

the value of that state is 63.10530471801758 

 we reached the state s5 which is the trap 

in the episode 2 : 

the action chosen for s1 is right 

the value of that state is 59.87788772583008 

the action chosen for s2 is right 

the value of that state is 61.97373962402344 

the action chosen for s3 is right 

the value of that state is 81.27578735351562 

the action chosen for s3 is right 

the value of that state is 81.27578735351562 

the action chosen for s3 is right 

the value of that state is 81.27578735351562 

the action chosen for s3 is up 

the value of that state is 81.27578735351562 

the action chosen for s6 is up 

the value of that state is 100.03924560546875 

 we reached the state s9 which is the goal 

in the episode 3 : 

the action chosen for s1 is right 

the value of that state is 59.87788772583008 

the action chosen for s2 

In [659]:

for start_state in env.states:

    for episode in range(10):
        
        env.current = start_state
        print(f"in the episode {episode + 1} for start {env.current.name} : \n")
        for step in range(10-1):
            s = env.current
            a,log_pi_a_of_s ,v = agent.act(s)
            print(f"the action chosen for {s.name} is {a} \n")
            print(f"the value of that state is {v} \n")

            exp = env.step(a)

            if exp["done"] : 
                if exp["next_state"].symbol == "X" : symbol = "trap" 
                else: symbol = "goal"
                print(f" we reached the state {exp["next_state"].name} which is the {symbol} \n")
                break

in the episode 1 for start s1 : 

the action chosen for s1 is right 

the value of that state is 59.87788772583008 

the action chosen for s2 is right 

the value of that state is 61.97373962402344 

the action chosen for s3 is up 

the value of that state is 81.27578735351562 

the action chosen for s6 is up 

the value of that state is 100.03924560546875 

 we reached the state s9 which is the goal 

in the episode 2 for start s1 : 

the action chosen for s1 is right 

the value of that state is 59.87788772583008 

the action chosen for s2 is right 

the value of that state is 61.97373962402344 

the action chosen for s3 is up 

the value of that state is 81.27578735351562 

the action chosen for s6 is up 

the value of that state is 100.03924560546875 

 we reached the state s9 which is the goal 

in the episode 3 for start s1 : 

the action chosen for s1 is right 

the value of that state is 59.87788772583008 

the action chosen for s2 is up 

the value of that state is 61.97373962

In [660]:
print("=== Value estimates after training ===")
for s in env.states:
    with torch.no_grad():
        x = torch.as_tensor(state_to_vector(s), dtype=torch.float32)
        _, v = agent.policy.net(x)
    print(f"  V({s.name}) = {v.item():.3f}  symbol={s.symbol}")

=== Value estimates after training ===
  V(s1) = 59.878  symbol=None
  V(s2) = 61.974  symbol=None
  V(s3) = 81.276  symbol=None
  V(s4) = 63.105  symbol=None
  V(s5) = 82.128  symbol=X
  V(s6) = 100.039  symbol=None
  V(s7) = 82.438  symbol=None
  V(s8) = 99.945  symbol=None
  V(s9) = 98.253  symbol=O


In [661]:

print(Actions)
for s in States2:
    probs , value = agent.policy.action_distribution(s)
    print(f"for the state {s.name} this is the action for each action: {probs}")

['up', 'right', 'down', 'left']
for the state s1 this is the action for each action: tensor([0.4393, 0.5026, 0.0217, 0.0364], grad_fn=<SoftmaxBackward0>)
for the state s2 this is the action for each action: tensor([0.1382, 0.8303, 0.0121, 0.0193], grad_fn=<SoftmaxBackward0>)
for the state s3 this is the action for each action: tensor([0.6214, 0.3756, 0.0015, 0.0015], grad_fn=<SoftmaxBackward0>)
for the state s4 this is the action for each action: tensor([0.7656, 0.2099, 0.0087, 0.0158], grad_fn=<SoftmaxBackward0>)
for the state s5 this is the action for each action: tensor([0.2753, 0.7229, 0.0008, 0.0010], grad_fn=<SoftmaxBackward0>)
for the state s6 this is the action for each action: tensor([5.2370e-01, 4.7617e-01, 6.6827e-05, 5.9297e-05],
       grad_fn=<SoftmaxBackward0>)
for the state s7 this is the action for each action: tensor([1.5043e-01, 8.4809e-01, 6.0280e-04, 8.7762e-04],
       grad_fn=<SoftmaxBackward0>)
for the state s8 this is the action for each action: tensor([2.4607e

In [662]:
net = ActorCritic(
    n_features=6,
    n_actions=n_actions,
    hidden_dim=64
)

pi = Policy_PPO(net)

optimizer = torch.optim.Adam(
    pi.net.parameters(),
    lr=3e-4
)

ppo = PPO(
    optimizer=optimizer,
    gamma=0.99,
    lam=0.95,
    clip_epsilon=0.2
)
###

TypeError: Policy_PPO.__init__() missing 1 required positional argument: 'actions'

### Potential issue

The only thing to be careful about is that everyone must reference the same network object.

#### Good:
```python
net = ActorCritic(...)

policy = Policy_PPO(net)
optimizer = torch.optim.Adam(net.parameters())

ppo = PPO(network=net, optimizer=optimizer)
```

All three share the same net.

#### Bad:
```python
policy = Policy_PPO(ActorCritic(...))
ppo = PPO(network=ActorCritic(...), ...)
``` 
Now there are two different networks:

policy uses one,
PPO updates another.

The policy will never see the learned weights.

### Recommendation

Pass the same network instance everywhere:
```python
net = ActorCritic(...)

policy = Policy_PPO(net)
optimizer = torch.optim.Adam(net.parameters())
strategy = PPO(net, optimizer)
agent = Agent(strategy, policy)
```
This architecture is clean, modular, and very close to how RL libraries (e.g. Stable-Baselines3, CleanRL) organize actor-critic agents.

In [ ]:
# import the sb3 here and see how to integrate them with my code (see if i can package it under my designed classes)


In [ ]:
# import the gymnasium stuff here